# 1. Split loading: CGMES files -> RDF database -> IIDM network

Importing CGMES is two jobs in one. The first is parsing: a few megabytes of RDF/XML become triples. The second is
conversion: those triples become an IIDM network. The first job is the expensive one, and it is repeated every time
the same grid model is loaded.

`pypowsybl.network` splits them. `RdfDatabase.load_cgmes` parses the instance files **once** into a SPARQL graph
database, and `from_rdf_db` builds a network out of the database afterwards. The network is the one the files
themselves would have produced.

Every call names a **scenario**: the base grid model the data belongs to, in practice a day. It is required and
never guessed, because a database is expected to hold many days side by side - which is what notebook 2 shows.

In [ ]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import matplotlib.pyplot as plt
import pypowsybl as pp

from notebook_utils import CGMES_ZIP, SCENARIO, connect, credentials, database_url, fresh_scenario

pp.set_config_read(False)
print(pp.__version__, '->', database_url())

## Connect

`connect` is a context manager. `memory:<name>` is an in-process store that needs no server; set
`PYPOWSYBL_RDF_DB=http://localhost:3030/ds` to run the same notebook against a real Fuseki (see the README).

In [ ]:
db = connect()
fresh_scenario(db, SCENARIO)
db

## Upload: the expensive half, done once

With a `version` the files become the **root snapshot** of the scenario, and the call returns the identifiers of
the CGMES models that were stored. Without one they are stored un-versioned and the call returns graph names.

In [ ]:
stored = db.load_cgmes(CGMES_ZIP, SCENARIO, '1.0')
stored

## What the database now holds

In [ ]:
db.scenarios()

In [ ]:
db.graphs(SCENARIO)

In [ ]:
db.models(SCENARIO)[['subset', 'kind', 'triple_count', 'chain_depth']]

In [ ]:
db.snapshots(SCENARIO)[['version', 'timestep', 'timestep_label', 'kind', 'depth', 'has_full']]

## The same network, both ways

`from_rdf_db` addresses a snapshot by `(scenario, version, timestep)`. Here the scenario holds exactly one, so
`'1.0'` and the base timestep are enough.

In [ ]:
n_db = pp.network.from_rdf_db(db, SCENARIO, '1.0')
n_file = pp.network.load(CGMES_ZIP)

print(n_db.id == n_file.id, len(n_db.get_loads()), len(n_file.get_loads()))

In [ ]:
def frame(n, getter, columns):
    # A real SPARQL server hands its statements back in its own index order, so the rows of the two frames are
    # the same set but not necessarily the same sequence; the comparison is on values, not on order.
    return getter(n)[columns].sort_index()

pd.testing.assert_frame_equal(frame(n_file, pp.network.Network.get_loads, ['p0', 'q0']),
                              frame(n_db, pp.network.Network.get_loads, ['p0', 'q0']))
pd.testing.assert_frame_equal(frame(n_file, pp.network.Network.get_generators, ['target_p', 'target_q', 'target_v']),
                              frame(n_db, pp.network.Network.get_generators, ['target_p', 'target_q', 'target_v']))
print('setpoints identical')

On the in-process backend the two networks are identical down to the serialised document. Against a real SPARQL
server they are equal in every value but may number the nodes of a voltage level differently, because that
numbering follows the order the conversion walks the query results, which is a property of the server's index and
not of the data.

In [ ]:
sorted_xiidm = {'iidm.export.xml.sorted': 'true'}
if database_url().startswith('memory:'):
    print(n_file.save_to_string('XIIDM', sorted_xiidm) == n_db.save_to_string('XIIDM', sorted_xiidm))
else:
    print('skipped: node numbering is server-dependent')

## What it costs

In [ ]:
def timed(call, runs=3):
    call()
    samples = []
    for _ in range(runs):
        start = time.perf_counter()
        call()
        samples.append((time.perf_counter() - start) * 1000)
    return sorted(samples)[len(samples) // 2]

file_ms = timed(lambda: pp.network.load(CGMES_ZIP))
warm_ms = timed(lambda: pp.network.from_rdf_db(db, SCENARIO, '1.0'))

def cold():
    with pp.network.connect(database_url(), cache=False, **credentials()) as fresh:
        pp.network.from_rdf_db(fresh, SCENARIO, '1.0')

cold_ms = timed(cold)
pd.Series({'file import': file_ms, 'db, cold': cold_ms, 'db, warm': warm_ms}, name='ms')

In [ ]:
ax = pd.Series({'file import': file_ms, 'db, cold': cold_ms, 'db, warm': warm_ms}).plot.bar(
    rot=0, color=['#888', '#4c72b0', '#55a868'])
ax.set_ylabel('milliseconds')
ax.set_title('Loading one grid model')
plt.tight_layout()

The database path wins once the graphs are cached - and the bigger the model, the more it wins, because the part it
skips is the parsing. On this ~1 MB fixture the margin is modest; on a 14 MB model the core benchmarks measure a
factor of three.

## Where the network thinks it is

In [ ]:
n_db.rdf_db_identity()

In [ ]:
db.close()